# NOVA Unified Intelligence Research\nResearch-only notebook: XGBoost + genuine PyTorch LSTM + SMC + immutable NOVA Brain replay + OOF meta-decision. CPU-first for Colab/Kaggle.\n\n**No training runs by default.** Do not enable training until a provenance-verified dataset, session-aware coverage receipt and broker-cost fixture are approved. This notebook cannot submit orders.\n\nFinal OOS starts at `2026-07-01T00:00:00Z`; development tuning must remain strictly before that boundary.

In [ ]:
from pathlib import Path\nimport os, json, hashlib, random, warnings\nimport numpy as np\nimport pandas as pd\nSEED = 42\nrandom.seed(SEED); np.random.seed(SEED)\nDATA_PATH = Path('/content/drive/MyDrive/NOVA/approved-market-data/XAUUSDm_H1.csv')\nMANIFEST_PATH = DATA_PATH.with_suffix('.manifest.json')\nOUTPUT_DIR = Path('/content/drive/MyDrive/NOVA/research-runs/unified-intelligence')\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\nRUN_TRAINING = False  # explicit opt-in only after all gates are reviewed\nRUN_LSTM = False\nFINAL_OOS_START = pd.Timestamp('2026-07-01T00:00:00Z')\nHORIZON = 5\nN_SPLITS = 5\nprint({'training_enabled': RUN_TRAINING, 'data_exists': DATA_PATH.is_file(), 'manifest_exists': MANIFEST_PATH.is_file()})

## 1. Dataset admission — fail closed\nManifest must identify genuine broker/provider source, exact symbol/timeframe, UTC half-open coverage, tick-volume semantics, gap receipt, row count, content SHA-256 and all read-only flags. Missing costs are not treated as zero.

In [ ]:
from workers.quant_engine.unified_evaluation import (\n    validate_dataset_manifest, expanding_purged_walk_forward,\n    validate_point_in_time_rows, require_immutable_llm_replay, fuse_research_signals,\n)\ndef load_approved_dataset(data_path=DATA_PATH, manifest_path=MANIFEST_PATH):\n    if not data_path.is_file() or not manifest_path.is_file():\n        raise RuntimeError('BLOCKED: approved dataset or manifest is absent; do not train on synthetic data.')\n    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))\n    digest = validate_dataset_manifest(manifest, data_path)\n    frame = pd.read_csv(data_path)\n    required = ['timestamp', 'open', 'high', 'low', 'close', 'tick_volume', 'spread']\n    missing = [c for c in required if c not in frame.columns]\n    if missing: raise ValueError(f'Missing source columns: {missing}')\n    frame['timestamp'] = pd.to_datetime(frame['timestamp'], utc=True, errors='raise')\n    validate_point_in_time_rows(frame['timestamp'].tolist())\n    for col in ['open','high','low','close','tick_volume','spread']:\n        frame[col] = pd.to_numeric(frame[col], errors='raise')\n    if not np.isfinite(frame[['open','high','low','close','tick_volume','spread']].to_numpy()).all():\n        raise ValueError('Non-finite market values')\n    if (frame.tick_volume < 0).any() or (frame.spread < 0).any(): raise ValueError('Negative volume/spread')\n    invalid = (frame.high < frame.low) | (frame.open < frame.low) | (frame.open > frame.high) | (frame.close < frame.low) | (frame.close > frame.high)\n    if invalid.any(): raise ValueError('Invalid OHLC bounds')\n    if len(frame) != int(manifest['row_count']): raise ValueError('Manifest row_count mismatch')\n    if frame.timestamp.min() < pd.Timestamp(manifest['actual_start_utc']) or frame.timestamp.max() >= pd.Timestamp(manifest['actual_end_utc']):\n        raise ValueError('Observed rows fall outside manifest UTC range')\n    # Gap receipt must be session-aware and explicitly PASS; no interpolation or synthetic candles.\n    if manifest.get('gap_validation', {}).get('status') != 'PASS': raise ValueError('BLOCKED: session-aware gap receipt is not PASS')\n    return frame, manifest, digest\nif DATA_PATH.is_file() and MANIFEST_PATH.is_file():\n    market, dataset_manifest, DATA_SHA256 = load_approved_dataset()\n    print('Dataset admitted:', len(market), DATA_SHA256)\nelse:\n    market = None\n    print('BLOCKED: no approved dataset artifact; training and performance evaluation are not run.')

## 2. Causal features and forward labels\nFeatures use only current/past closed bars. The decision is formed at bar close and the simulator enters no earlier than the next eligible bar. The last `HORIZON` rows have unresolved labels and are dropped, never coerced to class zero.

In [ ]:
def make_causal_features(frame):\n    x = pd.DataFrame(index=frame.index)\n    close = frame.close.astype(float)\n    ret = close.pct_change()\n    x['ret_1'] = ret\n    x['ret_5'] = close.pct_change(5)\n    x['volatility_20'] = ret.rolling(20, min_periods=20).std()\n    delta = close.diff()\n    gain = delta.clip(lower=0).rolling(14, min_periods=14).mean()\n    loss = (-delta.clip(upper=0)).rolling(14, min_periods=14).mean()\n    x['rsi_14'] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))\n    ema12 = close.ewm(span=12, adjust=False, min_periods=12).mean()\n    ema26 = close.ewm(span=26, adjust=False, min_periods=26).mean()\n    x['macd'] = ema12 - ema26\n    x['volume_ratio_20'] = frame.tick_volume / frame.tick_volume.rolling(20, min_periods=20).mean().replace(0, np.nan)\n    x['spread_points'] = frame.spread.astype(float)\n    return x.replace([np.inf, -np.inf], np.nan)\n\ndef make_forward_labels(frame, horizon=HORIZON):\n    future = frame.close.shift(-horizon)\n    label = (future > frame.close).astype('float')\n    label[future.isna()] = np.nan\n    return label\n\nif market is not None:\n    features = make_causal_features(market)\n    labels = make_forward_labels(market)\n    valid = features.notna().all(axis=1) & labels.notna()\n    research = market.loc[valid].copy()\n    X = features.loc[valid].copy()\n    y = labels.loc[valid].astype(int)\n    development_mask = research.timestamp < FINAL_OOS_START\n    oos_mask = research.timestamp >= FINAL_OOS_START\n    dev_X, dev_y = X.loc[development_mask], y.loc[development_mask]\n    oos_X, oos_y = X.loc[oos_mask], y.loc[oos_mask]\n    if dev_X.empty or oos_X.empty: raise RuntimeError('Development or sealed OOS partition is empty')\n    print('Development rows:', len(dev_X), '| sealed OOS rows:', len(oos_X))

## 3. Development-only walk-forward and base models\nEvery fold trains only on earlier rows; the forward-label horizon is purged before validation. Feature scaling, calibration and meta-learning are fit only inside development folds.

In [ ]:
def get_folds(n_rows):\n    return expanding_purged_walk_forward(n_rows, n_splits=N_SPLITS, min_train_size=max(500, n_rows//3), horizon=HORIZON)\n\ndef train_xgboost_fold(x_train, y_train, x_valid, y_valid):\n    from xgboost import XGBClassifier\n    model = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, eval_metric='logloss', random_state=SEED, n_jobs=2)\n    model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=False)\n    return model, model.predict_proba(x_valid)[:,1]\n\ndef train_lstm_fold(x_train, y_train, x_valid, y_valid, epochs=20, patience=4):\n    import torch\n    from torch import nn\n    torch.manual_seed(SEED)\n    class CausalLSTM(nn.Module):\n        def __init__(self, n_features, hidden=32):\n            super().__init__(); self.lstm=nn.LSTM(n_features, hidden, batch_first=True); self.dropout=nn.Dropout(0.1); self.head=nn.Linear(hidden,1)\n        def forward(self, z):\n            _, (h, _) = self.lstm(z); return self.head(self.dropout(h[-1])).squeeze(-1)\n    def windows(a, seq=32):\n        return np.stack([a[i-seq+1:i+1] for i in range(seq-1, len(a))])\n    # Caller must supply already scaled arrays; scaler is fit on training rows only.\n    tx, vx = windows(np.asarray(x_train, dtype=np.float32)), windows(np.asarray(x_valid, dtype=np.float32))\n    ty = np.asarray(y_train, dtype=np.float32)[31:]; vy = np.asarray(y_valid, dtype=np.float32)[31:]\n    if len(tx)==0 or len(vx)==0: raise ValueError('Insufficient sequence rows')\n    model=CausalLSTM(tx.shape[-1]); opt=torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4); loss_fn=nn.BCEWithLogitsLoss()\n    best=float('inf'); best_state=None; wait=0; history=[]\n    for epoch in range(epochs):\n        model.train(); opt.zero_grad(); loss=loss_fn(model(torch.from_numpy(tx)), torch.from_numpy(ty)); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()\n        model.eval()\n        with torch.no_grad(): val=float(loss_fn(model(torch.from_numpy(vx)), torch.from_numpy(vy)).item())\n        history.append({'epoch':epoch+1,'validation_loss':val})\n        if val < best: best=val; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0\n        else: wait+=1\n        if wait>=patience: break\n    if best_state is None: raise RuntimeError('LSTM did not produce a valid checkpoint')\n    model.load_state_dict(best_state)\n    with torch.no_grad(): probs=torch.sigmoid(model(torch.from_numpy(vx))).numpy()\n    import io\n    buf=io.BytesIO(); torch.save(best_state,buf); checkpoint=buf.getvalue()\n    return model, probs, history, hashlib.sha256(checkpoint).hexdigest()\n\nprint('Walk-forward folds are generated only after dataset admission; no fitting occurs in this cell.')

## 4. SMC and NOVA Brain replay contracts\nSMC must be the canonical confirmed-candle implementation; the 4H bias is available only after the 4H candle closes and pivots only after right-side confirmation. NOVA Brain contributes only immutable historical responses that existed by the decision time. No LLM call is made during historical replay.

In [ ]:
def validate_smc_event(event, decision_time):\n    required = ['event_time','available_at','signal','source_version','confirmed']\n    if any(k not in event for k in required): raise ValueError('SMC event provenance incomplete')\n    if not event['confirmed'] or pd.Timestamp(event['available_at']) > pd.Timestamp(decision_time):\n        raise ValueError('SMC pivot/HTF event was not yet available at decision time')\n    if event['signal'] not in (-1,0,1): raise ValueError('Invalid SMC signal')\n    return event\n\ndef validate_brain_replay(record, decision_time):\n    return require_immutable_llm_replay(record, pd.Timestamp(decision_time))\n\n# Load SMC and NOVA Brain ledgers only from separately approved, immutable artifacts.\nSMC_LEDGER_PATH = OUTPUT_DIR / 'smc_events.jsonl'\nBRAIN_LEDGER_PATH = OUTPUT_DIR / 'brain_replay.jsonl'\nprint('No SMC/LLM replay is inferred when the immutable ledgers are absent.')

## 5. Meta Decision Layer — OOF only\nTrain a meta-model on out-of-fold development predictions only. Never train it on in-sample base-model predictions. The final OOS is opened only after base models, calibrators, thresholds and meta-model are frozen. Missing, stale or contradictory evidence means ABSTAIN.

In [ ]:
from workers.quant_engine.unified_evaluation import fuse_research_signals\ndef build_meta_features(xgb_p, lstm_p, smc_signal, brain_signal):\n    return np.column_stack([xgb_p, lstm_p, smc_signal, brain_signal])\n\ndef fit_meta_model(oof_features, oof_labels):\n    from sklearn.linear_model import LogisticRegression\n    model = LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000, random_state=SEED)\n    model.fit(oof_features, oof_labels)\n    return model\n\ndef meta_decision(meta_model, xgb_p, lstm_p, smc_signal, brain_signal):\n    row = build_meta_features([xgb_p],[lstm_p],[smc_signal],[brain_signal])\n    p = float(meta_model.predict_proba(row)[0,1])\n    return fuse_research_signals(xgb_p,lstm_p,smc_signal,brain_signal,p,min_meta_probability=0.60)\n\nprint('Meta-model fit is prohibited until OOF development predictions have been generated and audited.')

## 6. Shared cost-aware evaluator\nAll candidates must use the same next-eligible-bar fill, symbol specification and cost assumptions. The evaluator must reject missing spread/commission/swap/slippage fixtures rather than treating unknown costs as zero. Bar-only data cannot claim tick-level fill accuracy.

In [ ]:
def net_trade_pnl(gross_pnl, spread_cost, commission, swap, slippage):\n    costs = [gross_pnl, spread_cost, commission, swap, slippage]\n    if not np.isfinite(costs).all(): raise ValueError('Non-finite PnL/cost input')\n    if min(spread_cost, commission, slippage) < 0: raise ValueError('Costs cannot be negative')\n    return float(gross_pnl - spread_cost - commission - swap - slippage)\n\ndef summarize_trade_ledger(trades):\n    if not trades: return {'closed_trades':0,'net_pnl':0.0,'profit_factor':None,'max_drawdown':None}\n    net = np.asarray([float(t['net_pnl']) for t in trades], dtype=float)\n    if not np.isfinite(net).all(): raise ValueError('Invalid trade ledger')\n    gains, losses = net[net>0].sum(), -net[net<0].sum()\n    equity = np.cumsum(net); peak = np.maximum.accumulate(np.r_[0.0,equity])[1:]\n    dd = peak - equity\n    return {'closed_trades':int(len(net)),'net_pnl':float(net.sum()),'profit_factor':float(gains/losses) if losses>0 else None,'max_drawdown':float(dd.max()) if len(dd) else 0.0,'expectancy_per_trade':float(net.mean())}\n\nprint('Cost model must be populated from versioned Exness XAUUSDm Demo specifications before any strategy replay.')

## 7. Explicit training gate and run receipt\nThe notebook will not fit a model, evaluate OOS, or write a model artifact while data admission is blocked. Set `RUN_TRAINING=True` only after reviewing the exact dataset manifest and costs. This is not permission to promote or trade.

In [ ]:
if not RUN_TRAINING:\n    print('BLOCKED BY DEFAULT: set RUN_TRAINING=True only after independent data and cost review.')\nelif market is None:\n    raise RuntimeError('Training refused: approved dataset was not admitted.')\nelse:\n    raise NotImplementedError('Execution orchestration is intentionally gated pending verified OOF model adapters, SMC/Brain ledgers and broker-cost fixtures.')\n\nRUN_RECEIPT = {\n    'status': 'BLOCKED' if market is None or not RUN_TRAINING else 'PENDING_ADAPTERS',\n    'dataset_sha256': None if market is None else DATA_SHA256,\n    'final_oos_start_utc': FINAL_OOS_START.isoformat(),\n    'training_enabled': RUN_TRAINING,\n    'ordersEnabled': False, 'executionEnabled': False, 'liveTradingEnabled': False,\n}\n(OUTPUT_DIR / 'run_receipt.json').write_text(json.dumps(RUN_RECEIPT, indent=2), encoding='utf-8')\nprint(RUN_RECEIPT)